# P9 SD 1.5 butterfly LoRA: Kaggle runner

Thin wrapper: all logic lives in `src/sdlora`.

Settings: **GPU T4, Internet ON**, then *Save Version -> Save & Run All*. Runs are resumable (a finished
(variant, seed) is skipped; trained LoRAs are reused). The smoke config runs with `--check`, so the notebook stops if
a run is missing, a metric is not finite or the LoRA does not change the images.

Download `p9_results.zip` from the Output tab: results, bootstrap, safety and training tables, per-prompt metrics,
training loss logs and small sample grids. Generated images, the dataset copy, feature caches and LoRA weights stay
out of the zip (size). Data: Smithsonian butterflies, CC0-1.0. Model: SD 1.5, CreativeML OpenRAIL-M.

In [ ]:
!git clone --depth 1 https://github.com/JDevender007/Artificial_Intelligence.git
%cd Artificial_Intelligence/09-butterfly-lora
!pip install -q diffusers peft accelerate torchmetrics torch-fidelity && pip install -q -e . --no-deps
!pip uninstall -y -q torchao  # peft refuses to build LoRA layers next to Kaggle's torchao 0.10

In [ ]:
!mkdir -p /kaggle/working/outputs
!python -c "import torch, transformers, diffusers, peft, accelerate, torchmetrics, numpy, pandas, PIL, yaml; [print(m.__name__, m.__version__) for m in (torch, transformers, diffusers, peft, accelerate, torchmetrics, numpy, pandas, PIL, yaml)]; print('cuda', torch.cuda.is_available(), torch.version.cuda)" | tee /kaggle/working/outputs/env.txt
!(nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo no-gpu) | tee -a /kaggle/working/outputs/env.txt
!pip freeze | grep -iE '^(torch|transformers|accelerate|tokenizers|safetensors|huggingface.hub|diffusers|peft|torchmetrics|torch.fidelity|numpy|pandas|pillow|pyyaml)==' | tee /kaggle/working/outputs/versions.txt

## Run `configs/smoke.yaml`

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, 'scripts/run_experiment.py', '--config', 'configs/smoke.yaml', '--check', '--set', 'output_dir=/kaggle/working/outputs/smoke'], check=True)

In [ ]:
from pathlib import Path

for f in ['results.md', 'bootstrap.md', 'safety.md', 'training.md']:
    p = Path('/kaggle/working/outputs/smoke') / f
    print(f'===== {f}')
    print(p.read_text() if p.exists() else '(missing)')

## Run `configs/main.yaml`

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, 'scripts/run_experiment.py', '--config', 'configs/main.yaml', '--check', '--set', 'output_dir=/kaggle/working/outputs/main'], check=True)

In [ ]:
from pathlib import Path

for f in ['results.md', 'bootstrap.md', 'safety.md', 'training.md']:
    p = Path('/kaggle/working/outputs/main') / f
    print(f'===== {f}')
    print(p.read_text() if p.exists() else '(missing)')

## Package results

In [ ]:
%cd /kaggle/working
!zip -qr p9_results.zip outputs -x '*/data/*' '*/gen/*' '*/features/*' '*.safetensors'
!rm -rf outputs Artificial_Intelligence && ls -la /kaggle/working